# T302 — Train v1 (BGE-m3 fine-tune on catalog + vocab pairs)

Phase 3 compute room #2 (per ADR 012). Runs on **Google Colab Free T4**, ~30–60 min walk-away.

**What it does:**
1. Pulls the v2 candidate pair set (27,172 pairs) from private HF Datasets repo `ManmohanBuildsProducts/auto-parts-search-pairs`.
2. Filters to `label == 1.0` positives (MultipleNegativesRanking only uses positives; in-batch negatives are sampled automatically).
3. Fine-tunes `BAAI/bge-m3` for 1 epoch with MNR loss, batch 16, lr 2e-5, fp16 AMP.
4. Pushes the trained model to private HF repo `ManmohanBuildsProducts/auto-parts-search-v1`.

**Before running:**
- Runtime → Change runtime type → T4 GPU.
- Have your HF token ready (with *write* scope on your namespace).

**Memory note:** BGE-m3 is 568M params. On a 15GB T4 we use batch 16 + seq 128 + fp16 AMP (peak ~8GB). Batch 32 / seq 256 OOMs.

## 1. Install + auth

In [ ]:
!pip install -q sentence-transformers==3.4.1 datasets==3.2.0 huggingface_hub==0.27.0

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Pull training pairs from HF

In [ ]:
from datasets import load_dataset

PAIRS_REPO = 'ManmohanBuildsProducts/auto-parts-search-pairs'
ds = load_dataset(PAIRS_REPO, split='train')
print(f'loaded {len(ds)} pairs')

positives = ds.filter(lambda r: r['label'] >= 1.0)
print(f'positives (label==1.0): {len(positives)}')

from collections import Counter
print('by pair_type:', Counter(positives['pair_type']).most_common())

## 3. Build InputExamples + shuffle

In [ ]:
import random
from sentence_transformers import InputExample

SEED = 42
random.seed(SEED)

examples = [InputExample(texts=[r['text_a'], r['text_b']]) for r in positives]
random.shuffle(examples)
print(f'{len(examples)} training examples')

## 4. Load BGE-m3 base

In [ ]:
import torch
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

BASE = 'BAAI/bge-m3'
print(f'CUDA available: {torch.cuda.is_available()}, device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}')

model = SentenceTransformer(BASE)
model.max_seq_length = 128

BATCH = 16
loader = DataLoader(examples, shuffle=True, batch_size=BATCH)
train_loss = losses.MultipleNegativesRankingLoss(model)

EPOCHS = 1
WARMUP = int(len(loader) * 0.1)
print(f'steps/epoch: {len(loader)}, warmup: {WARMUP}')

## 5. Train

In [ ]:
OUT_DIR = 'auto-parts-search-v1'

model.fit(
    train_objectives=[(loader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=WARMUP,
    optimizer_params={'lr': 2e-5},
    output_path=OUT_DIR,
    show_progress_bar=True,
    use_amp=True,
)
print('training done.')

## 6. Push to HF Hub (private)

In [ ]:
from huggingface_hub import HfApi

MODEL_REPO = 'ManmohanBuildsProducts/auto-parts-search-v1'
api = HfApi()
api.create_repo(MODEL_REPO, private=True, exist_ok=True)  # idempotent — avoids 409 on reruns
api.upload_folder(
    folder_path=OUT_DIR,
    repo_id=MODEL_REPO,
    commit_message=f'v1.1 — label-corrected, {len(examples)} positives, 1 epoch, MNR',
)
print(f'pushed to https://huggingface.co/{MODEL_REPO}')

## 7. Sanity check — encode a few queries

In [ ]:
probes = ['patti badal do', 'brake pad Maruti Swift', 'kick starter', 'engine noise abnormal']
emb = model.encode(probes, normalize_embeddings=True)
print('embedding shape:', emb.shape)
print('sanity: identical query cosine self =', float((emb[0] * emb[0]).sum()))

## Next steps (locally)

Back on the laptop:
```bash
python3.11 -m training.evaluate \
    --model ManmohanBuildsProducts/auto-parts-search-v1 \
    --benchmark data/training/golden/benchmark_dev.json \
    --out data/training/experiments/2026-04-13-kg-pairs/v1-dev.json
```

Decision gate: if MRR on dev beats BGE-m3 baseline (0.391) by ≥10%, proceed to T208b + v2. If not, stop and revisit pair generation — do **not** tune hyperparameters (budget guardrail per Phase 3 plan).